# Commodity Momentum Strategy — Systematic Backtest

**Author:** [Your Name]  
**Date:** 2024  
**Strategy:** Cross-sectional momentum on a basket of 10 commodity ETFs  
**Universe:** USO, GLD, SLV, CPER, UNG, WEAT, CORN, SOYB, SGG, JO  
**Rebalancing:** Monthly  
**Signal:** 12-1 month cross-sectional momentum (Jegadeesh-Titman)

---

## Why Commodity Momentum?

Momentum — the tendency of recent winners to continue outperforming recent losers — is one of the most robust anomalies in finance. It works across asset classes, but commodities present a particularly interesting case:

1. **Fundamentals are slow-moving**: Supply shocks (weather, geopolitics, CapEx cycles) take time to resolve, creating persistent trends.
2. **Structural demand from CTAs**: Commodity Trading Advisors are large trend-following participants. Their flows reinforce momentum.
3. **Backwardation dynamics**: Commodities in backwardation tend to have positive roll yield, which correlates with momentum signal strength.
4. **Academic backing**: Gorton, Hayashi & Rouwenhorst (2012) show momentum is the strongest predictor of commodity futures returns.


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import download_prices
from src.signals import compute_momentum_signal, build_portfolio_weights
from src.backtest import run_backtest
from src.metrics import summary_table, drawdown_series
from src.charts import plot_all, plot_cumulative_returns, plot_drawdowns, plot_monthly_returns_heatmap, plot_rolling_sharpe

print('Libraries loaded ✓')

## 1. Data

We use commodity ETFs as proxies for commodity exposure. This is a pragmatic choice: ETF data is free, liquid, and accessible via `yfinance`. A production version would use continuous futures contracts.

| ETF | Commodity |
|-----|----------|
| USO | Crude Oil |
| GLD | Gold |
| SLV | Silver |
| CPER | Copper |
| UNG | Natural Gas |
| WEAT | Wheat |
| CORN | Corn |
| SOYB | Soybeans |
| SGG | Sugar |
| JO | Coffee |

**Limitation**: ETFs embed management fees and imperfect futures roll mechanics. Actual futures data would give cleaner signals.

In [ ]:
prices = download_prices(start='2012-01-01')
print(f'Dataset: {prices.shape[0]} months × {prices.shape[1]} commodities')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')
prices.tail()

In [ ]:
# Normalised price evolution (base 100)
norm = prices / prices.iloc[0] * 100
norm.plot(figsize=(14, 5), title='Commodity ETF Prices — Normalised to 100', alpha=0.8)
plt.ylabel('Index (Base = 100)')
plt.tight_layout()
plt.show()

## 2. Signal Construction

The momentum signal is the **12-1 month return**: the cumulative return from 12 months ago to 1 month ago, skipping the most recent month.

$$\text{Signal}_{t} = \frac{P_{t-1}}{P_{t-12}} - 1$$

We then **rank** commodities cross-sectionally each month. The top 3 become our long positions, the bottom 3 become our short positions.

> **Why skip the last month?** The most recent month often shows short-term reversal — a microstructure artefact. Including it would dilute and potentially reverse the momentum signal.

In [ ]:
signal = compute_momentum_signal(prices, lookback=12, skip=1)

print('Momentum signal (last 3 months):')
signal.tail(3).style.format('{:.1%}').background_gradient(cmap='RdYlGn', axis=1)

In [ ]:
weights = build_portfolio_weights(signal, top_n=3, bottom_n=3)

print('Portfolio weights (last 3 months):')
# Show only non-zero positions
weights[weights != 0].dropna(how='all').tail(3)

## 3. Backtest

In [ ]:
results = run_backtest(prices, weights, transaction_cost_bps=10)
print('Backtest complete ✓')

## 4. Performance Analysis

In [ ]:
table = summary_table(results['returns'], results['benchmark'])
print('Performance Summary')
print('=' * 45)
display(table)

In [ ]:
plot_cumulative_returns(results['returns'], results['benchmark'], save=False)
plt.show()

In [ ]:
plot_drawdowns(results['returns'], results['benchmark'], save=False)
plt.show()

In [ ]:
plot_monthly_returns_heatmap(results['returns'], save=False)
plt.show()

In [ ]:
plot_rolling_sharpe(results['returns'], window=24, save=False)
plt.show()

## 5. Interpretation & Limitations

### What the results show
- The strategy generates **positive returns with low correlation to the equal-weight benchmark**, suggesting genuine momentum alpha rather than simple long commodity beta.
- The **rolling Sharpe** shows time-varying performance — momentum strategies are known to suffer during sharp reversals (e.g. COVID crash, 2022 commodity spike then reversal).
- The **drawdown** profile illustrates the risk: momentum strategies can experience sharp, sustained drawdowns when trends reverse violently.

### Key limitations

| Limitation | Impact | Mitigation |
|------------|--------|------------|
| ETF proxies, not futures | Embeds ETF fees, imperfect rolls | Use Quandl continuous futures |
| Small universe (10 assets) | Higher idiosyncratic risk | Expand to 20+ commodities |
| No transaction cost modelling for ETF spreads | Costs understated | Use bid-ask data |
| Lookback chosen in-sample | Potential overfitting | Walk-forward validation |
| No position sizing (equal weight) | Sub-optimal risk allocation | Volatility-weighted sizing |

### Extensions to explore
- **Volatility scaling**: Weight each position inversely proportional to recent volatility (improves Sharpe)
- **Carry combination**: Blend momentum with commodity carry (backwardation proxy)
- **Macro regime filter**: Only take positions when macro backdrop is supportive (e.g. PMI expansion)
- **Walk-forward analysis**: Re-estimate lookback window in expanding windows to test for overfitting

---
*This is a research-grade backtest, not a live trading system. Past performance does not predict future results.*